# 🔄 Recurrence Relations & the Master Theorem

**Every divide-and-conquer algorithm has a recurrence hiding inside it. Learn to read the recurrence and know the complexity — without solving it.**

---

## The Hook
How many branches does an expanding tree have?

## ⚠️ Common Wrong Intuition
People often mistakenly add branches instead of multiplying them...

## 1. Concept Intuition

When MergeSort splits an array in half, sorts both halves, and merges:

$$T(n) = 2 \cdot T(n/2) + O(n)$$

This is a **recurrence relation** — it defines the work at size $n$ in terms of work at smaller sizes.

Reading it out loud: *"Solving a problem of size n costs two sub-problems of size n/2, plus n work to combine."*

### The three pieces

Every divide-and-conquer recurrence has three parameters:

| Parameter | Symbol | Meaning |
|-----------|--------|---------|
| Subproblems | $a$ | How many recursive calls? |
| Shrinkage | $b$ | By what factor does the input shrink? |
| Combine cost | $f(n)$ | How much work outside the recursion? |

$$T(n) = a \cdot T(n/b) + f(n)$$

### The Master Theorem (intuition)

The battle is between **how fast the tree grows** ($a$ subproblems) vs **how fast the work shrinks** ($n/b$ per level). The winner determines the complexity:

- **Leaves win** ($a > b^d$): Work is dominated by the exponentially many base cases → $O(n^{\log_b a})$
- **Tie** ($a = b^d$): Every level does the same work → $O(n^d \log n)$
- **Root wins** ($a < b^d$): Work at the top level dominates → $O(n^d)$

where $d$ is the exponent in $f(n) = O(n^d)$.

## 2. Visual Representation

Let's visualize the recursion tree to see where the work happens.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def plot_recursion_tree_work(a, b, d, n=64, max_depth=6):
    """Visualize work done at each level of a recursion tree.
    
    T(n) = a·T(n/b) + O(n^d)
    """
    depths = list(range(max_depth + 1))
    
    # At level k: a^k subproblems, each of size n/b^k
    # Work at level k: a^k × (n/b^k)^d = n^d × (a/b^d)^k
    work_per_level = [a**k * (n / b**k)**d for k in depths]
    subproblems = [a**k for k in depths]
    sizes = [n / b**k for k in depths]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Work per level
    colors = ['#f43f5e' if w == max(work_per_level) else '#6366f1' for w in work_per_level]
    axes[0].barh(depths, work_per_level, color=colors, alpha=0.8)
    axes[0].set_ylabel('Depth')
    axes[0].set_xlabel('Work at this level')
    axes[0].set_title('Work Distribution', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].grid(True, alpha=0.3)
    
    # Subproblems vs Size
    ax2 = axes[1]
    ax2.plot(depths, subproblems, 'o-', color='#f97316', linewidth=2, label='Subproblems (a^k)')
    ax2_twin = ax2.twinx()
    ax2_twin.plot(depths, sizes, 's-', color='#10b981', linewidth=2, label='Problem size (n/b^k)')
    ax2.set_xlabel('Depth')
    ax2.set_ylabel('Subproblems', color='#f97316')
    ax2_twin.set_ylabel('Problem size', color='#10b981')
    ax2.set_title('Growth vs Shrinkage', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Cumulative work
    cum_work = np.cumsum(work_per_level)
    axes[2].fill_between(depths, cum_work, color='#6366f1', alpha=0.3)
    axes[2].plot(depths, cum_work, 'o-', color='#6366f1', linewidth=2)
    axes[2].set_xlabel('Depth')
    axes[2].set_ylabel('Cumulative work')
    axes[2].set_title('Where does total work come from?', fontsize=12, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    
    # Determine case
    ratio = a / b**d
    if ratio > 1:
        case = f'LEAVES WIN (a={a} > b^d={b**d:.1f}) → O(n^{np.log(a)/np.log(b):.2f})'
    elif ratio == 1:
        case = f'TIE (a={a} = b^d={b**d:.1f}) → O(n^{d} log n)'
    else:
        case = f'ROOT WINS (a={a} < b^d={b**d:.1f}) → O(n^{d})'
    
    fig.suptitle(f'T(n) = {a}·T(n/{b}) + O(n^{d})  |  {case}', 
                fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# MergeSort: T(n) = 2T(n/2) + O(n)
plot_recursion_tree_work(a=2, b=2, d=1, n=64)

In [ ]:
# Compare all three Master Theorem cases
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cases = [
    (4, 2, 1, 'Leaves win: T(n)=4T(n/2)+O(n)'),   # a=4 > b^d=2
    (2, 2, 1, 'Tie: T(n)=2T(n/2)+O(n)'),           # a=2 = b^d=2 (MergeSort)
    (2, 4, 1, 'Root wins: T(n)=2T(n/4)+O(n)'),     # a=2 < b^d=4
]

for ax, (a, b, d, title) in zip(axes, cases):
    depths = list(range(7))
    work_per_level = [a**k * (64 / b**k)**d for k in depths]
    colors = ['#f43f5e' if w == max(work_per_level) else '#6366f1' for w in work_per_level]
    ax.barh(depths, work_per_level, color=colors, alpha=0.8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Work')
    ax.set_ylabel('Depth')

plt.suptitle('The Three Cases of the Master Theorem', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 3. Mathematical Formulation

### The Master Theorem

For recurrences of the form $T(n) = a \cdot T(n/b) + \Theta(n^d)$ where $a \geq 1$, $b > 1$:

$$T(n) = \begin{cases} \Theta(n^d) & \text{if } a < b^d \quad \text{(root dominates)} \\ \Theta(n^d \log n) & \text{if } a = b^d \quad \text{(balanced)} \\ \Theta(n^{\log_b a}) & \text{if } a > b^d \quad \text{(leaves dominate)} \end{cases}$$

### Famous examples

| Algorithm | Recurrence | a | b | d | Case | Complexity |
|-----------|-----------|---|---|---|------|------------|
| Binary search | $T(n) = T(n/2) + O(1)$ | 1 | 2 | 0 | Tie | $O(\log n)$ |
| MergeSort | $T(n) = 2T(n/2) + O(n)$ | 2 | 2 | 1 | Tie | $O(n \log n)$ |
| Karatsuba multiply | $T(n) = 3T(n/2) + O(n)$ | 3 | 2 | 1 | Leaves | $O(n^{1.585})$ |
| Strassen multiply | $T(n) = 7T(n/2) + O(n^2)$ | 7 | 2 | 2 | Leaves | $O(n^{2.807})$ |
| Naive matrix multiply | $T(n) = 8T(n/2) + O(n^2)$ | 8 | 2 | 2 | Leaves | $O(n^3)$ |

## 4. Code Implementation

### Empirically verifying the Master Theorem

In [ ]:
def solve_recurrence(a, b, d, n, base_cost=1):
    """Solve T(n) = a*T(n/b) + n^d by unrolling."""
    if n <= 1:
        return base_cost
    combine_cost = n**d
    sub_cost = sum(solve_recurrence(a, b, d, n/b, base_cost) for _ in range(a))
    return sub_cost + combine_cost

# Verify MergeSort: T(n) = 2T(n/2) + n → O(n log n)
sizes = [2**k for k in range(1, 11)]
merge_costs = [solve_recurrence(2, 2, 1, n) for n in sizes]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot actual vs theoretical
n_arr = np.array(sizes, dtype=float)
axes[0].plot(sizes, merge_costs, 'o-', color='#6366f1', linewidth=2, label='Actual T(n)')
# Fit: T(n) ≈ c × n log n
c = merge_costs[-1] / (sizes[-1] * np.log2(sizes[-1]))
axes[0].plot(n_arr, c * n_arr * np.log2(n_arr), '--', color='#10b981', 
             linewidth=2, label=f'{c:.1f} × n log₂ n')
axes[0].set_xlabel('n')
axes[0].set_ylabel('T(n)')
axes[0].set_title('MergeSort: T(n) = 2T(n/2) + n', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ratio T(n) / (n log n) should converge to a constant
ratios = [t / (n * np.log2(n)) if n > 1 else 0 for t, n in zip(merge_costs, sizes)]
axes[1].plot(sizes[1:], ratios[1:], 'o-', color='#f97316', linewidth=2)
axes[1].set_xlabel('n')
axes[1].set_ylabel('T(n) / (n log n)')
axes[1].set_title('Ratio converges → confirms O(n log n)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare three recurrences side by side
recurrences = [
    (1, 2, 0, 'Binary search: T(n)=T(n/2)+O(1)', 'O(log n)'),
    (2, 2, 1, 'MergeSort: T(n)=2T(n/2)+O(n)', 'O(n log n)'),
    (3, 2, 1, 'Karatsuba: T(n)=3T(n/2)+O(n)', 'O(n^1.58)'),
]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#10b981', '#6366f1', '#f43f5e']

for (a, b, d, label, bigO), color in zip(recurrences, colors):
    costs = [solve_recurrence(a, b, d, n) for n in sizes]
    ax.plot(sizes, costs, 'o-', color=color, linewidth=2, 
            label=f'{label} → {bigO}')

ax.set_xlabel('n', fontsize=12)
ax.set_ylabel('T(n)', fontsize=12)
ax.set_title('Master Theorem in Action', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Interactive Experiment

### Master Theorem Explorer

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

@interact(
    a=IntSlider(value=2, min=1, max=8, step=1, description='a (subprobs)'),
    b=IntSlider(value=2, min=2, max=4, step=1, description='b (shrink)'),
    d=FloatSlider(value=1.0, min=0, max=3, step=0.5, description='d (combine)')
)
def master_theorem_explorer(a, b, d):
    sizes_exp = [2**k for k in range(1, 9)]
    costs = [solve_recurrence(a, b, d, n) for n in sizes_exp]
    
    # Determine case
    critical = b**d
    if a < critical:
        case = f'ROOT WINS: O(n^{d})'
        theory = [n**d for n in sizes_exp]
    elif a == critical:
        case = f'TIE: O(n^{d} log n)'
        theory = [n**d * np.log2(n) for n in sizes_exp]
    else:
        exp = np.log(a) / np.log(b)
        case = f'LEAVES WIN: O(n^{exp:.2f})'
        theory = [n**exp for n in sizes_exp]
    
    # Normalize theory to match scale
    scale = costs[-1] / theory[-1] if theory[-1] > 0 else 1
    theory_scaled = [t * scale for t in theory]
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(sizes_exp, costs, 'o-', color='#6366f1', linewidth=2, label='Actual T(n)')
    ax.plot(sizes_exp, theory_scaled, '--', color='#10b981', linewidth=2, label=f'Theory: {case}')
    ax.set_xlabel('n', fontsize=12)
    ax.set_ylabel('T(n)', fontsize=12)
    ax.set_title(f'T(n) = {a}·T(n/{b}) + n^{d}  →  {case}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f'a = {a}, b = {b}, d = {d}')
    print(f'b^d = {critical:.1f}')
    print(f'a vs b^d: {a} {"<" if a < critical else "=" if a == critical else ">"} {critical:.1f}')

## 6. Tool Exploration

Understanding recursion depth and memoization.

In [ ]:
import sys

print(f'Default recursion limit: {sys.getrecursionlimit()}')
print()

# Fibonacci: the classic overlapping subproblems
call_count = [0]

def fib_naive(n):
    call_count[0] += 1
    if n <= 1: return n
    return fib_naive(n-1) + fib_naive(n-2)

# T(n) = T(n-1) + T(n-2) + O(1) → O(2^n) !
for n in [10, 15, 20, 25, 30]:
    call_count[0] = 0
    result = fib_naive(n)
    print(f'fib({n}) = {result:>10,}  |  calls: {call_count[0]:>12,}')

print()
print('Notice: calls roughly double each time n increases by 1!')
print('This is O(2^n) — the recurrence T(n) = T(n-1) + T(n-2) grows like Fibonacci itself.')
print()

# With memoization: O(n)
from functools import lru_cache

call_count_memo = [0]

@lru_cache(maxsize=None)
def fib_memo(n):
    call_count_memo[0] += 1
    if n <= 1: return n
    return fib_memo(n-1) + fib_memo(n-2)

call_count_memo[0] = 0
result = fib_memo(100)
print(f'fib_memo(100) = {result}  |  calls: {call_count_memo[0]}')
print('Memoization reduces O(2^n) to O(n) by caching results!')

## 7. Reflection Questions

1. Binary search has recurrence $T(n) = T(n/2) + O(1)$. Apply the Master Theorem: what are $a$, $b$, $d$? Which case? What's the complexity?

2. Strassen's matrix multiplication uses 7 sub-multiplications instead of 8. Write the recurrence for both. Why does this small difference (7 vs 8) matter so much?

3. The Fibonacci recurrence $T(n) = T(n-1) + T(n-2)$ doesn't fit the Master Theorem (why not?). What's the actual growth rate, and how does memoization change it?

4. In the interactive explorer, set $a=4, b=2, d=1$. Then try $a=4, b=2, d=2$. The complexity changes from $O(n^2)$ to $O(n^2 \log n)$. Why does doubling $d$ here change the case from "leaves win" to "tie"?

5. Design an algorithm with recurrence $T(n) = 9T(n/3) + O(n^2)$. What's its complexity? Now redesign to use 8 subproblems instead of 9. What's the new complexity? How much did you save?

---

*You can now read any divide-and-conquer recurrence and know its complexity. → Try the exercises: `02_exercises.ipynb`*

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*